# Avance 2 - Paso 1: Estructura y Calidad de los Datos
**Asignatura:** EIN092B Visualización de Datos  
**Profesor:** Jesús A. Parra  
**Proyecto:** Impacto de las Áreas Verdes en la Tasa de Hurtos (Región Metropolitana, 2023)  
**Integrantes:** Vicente Paredes, Tomás Armijo, Francisca Matamala  

---

## 🎯 Objetivo de este Notebook
En cumplimiento con la **Sección 3.1** de la pauta del Avance 2 (*Estructura y calidad de los datos*), este notebook documenta el proceso reproducible de:
1. **Auditoría de datos crudos:** Inspección de dimensiones, tipos de variables y valores faltantes en las 3 fuentes (MINVU, CEAD y SAE).
2. **Justificación de limpieza y filtros:** 
   - Horizonte temporal: año 2023 ($T$).
   - Alcance territorial: 52 comunas de la Región Metropolitana.
   - Tipo de delito: delimitado a *Hurtos* ($Y$).
3. **Integración territorial:** Unión mediante el Código Único Territorial (`CUT`) oficial.
4. **Ingeniería de variables normalizadas:** Resolución del sesgo de población mediante:
   - $X$: Metros cuadrados de áreas verdes por habitante ($m^2/\text{hab}$).
   - $Y$: Tasa de hurtos por cada 100.000 habitantes.
   - Variable de control contextual: Porcentaje de pobreza por ingresos.
5. **Exportación:** Guardado de los datasets limpios en `data/processed/` para el análisis y la aplicación en Streamlit.

In [1]:
import os
import pandas as pd
import numpy as np

# Configuración de visualización en pandas
pd.set_option('display.max_columns', 25)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Definición de rutas relativas del repositorio
RAW_DIR = os.path.join('..', 'data', 'raw')
PROC_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROC_DIR, exist_ok=True)

print('Directorio de datos crudos:', os.path.abspath(RAW_DIR))
print('Directorio de datos procesados:', os.path.abspath(PROC_DIR))

Directorio de datos crudos: /home/r0ars/Documents/Impacto-de-las-Areas-Verdes-en-la-Tasa-de-Hurtos-2023/data/raw
Directorio de datos procesados: /home/r0ars/Documents/Impacto-de-las-Areas-Verdes-en-la-Tasa-de-Hurtos-2023/data/processed


---
## 1. Dataset Áreas Verdes (IDE MINVU - INE)
* **Archivo:** `Areas_Verdes_Minvu_INE.csv`
* **Unidad de observación cruda:** Un polígono o espacio público catastrado (plaza, parque, etc.).
* **Objetivo de limpieza:** Filtrar las áreas verdes pertenecientes a la Región Metropolitana y agregarlas a nivel comunal (`CUT`).

In [2]:
# 1.1 Carga e inspección inicial
path_av = os.path.join(RAW_DIR, 'Areas_Verdes_Minvu_INE.csv')
df_av_raw = pd.read_csv(path_av, low_memory=False)
print(f'Dimensiones crudas a nivel nacional: {df_av_raw.shape[0]} filas, {df_av_raw.shape[1]} columnas.')

# 1.2 Filtro territorial: Región Metropolitana (códigos CUT inician con '13')
df_av_rm = df_av_raw[df_av_raw['CUT'].astype(str).str.startswith('13')].copy()
print(f'Áreas verdes pertenecientes a la Región Metropolitana: {len(df_av_rm)} registros.')
print(f'Superficie total en la RM: {df_av_rm["Shape_Area"].sum():,.2f} m².')

# 1.3 Agregación a nivel comunal
df_av_comunal = df_av_rm.groupby('CUT', as_index=False).agg(
    m2_areas_verdes=('Shape_Area', 'sum'),
    cantidad_espacios_publicos=('OBJECTID_1', 'count'),
    cantidad_plazas=('TIPO_EP', lambda s: (s == 'PLAZA').sum()),
    cantidad_parques=('TIPO_EP', lambda s: (s == 'PARQUE').sum())
)
df_av_comunal['CUT'] = df_av_comunal['CUT'].astype(int)

print(f'Total comunas con áreas verdes catastradas: {len(df_av_comunal)}')
df_av_comunal.head()

Dimensiones crudas a nivel nacional: 18173 filas, 20 columnas.
Áreas verdes pertenecientes a la Región Metropolitana: 8160 registros.
Superficie total en la RM: 38,412,044.44 m².
Total comunas con áreas verdes catastradas: 52


---
## 2. Dataset Delictual (CEAD - Subsecretaría de Prevención del Delito)
* **Archivo:** `cead_delincuencia_chile.parquet`
* **Unidad de observación cruda:** Registros mensuales de casos policiales por tipo de delito y comuna.
* **Objetivo de limpieza:** Filtrar el año 2023 ($T$), la Región Metropolitana y acotar al delito de *Hurtos* ($Y$).

In [3]:
# 2.1 Carga e inspección inicial
path_cead = os.path.join(RAW_DIR, 'cead_delincuencia_chile.parquet')
df_cead_raw = pd.read_parquet(path_cead)
print(f'Dimensiones crudas históricas: {df_cead_raw.shape[0]:,} filas, {df_cead_raw.shape[1]} columnas.')

# 2.2 Filtro temporal (2023), regional (RM) y delictual (Hurtos)
df_cead_raw['fecha'] = pd.to_datetime(df_cead_raw['fecha'])
df_cead_rm_2023 = df_cead_raw[
    (df_cead_raw['fecha'].dt.year == 2023) & 
    (df_cead_raw['cut_region'] == 13.0) & 
    (df_cead_raw['delito'] == 'Hurtos')
].copy()

total_casos_hurtos = df_cead_rm_2023['delito_n'].sum()
print(f'Total de casos policiales por Hurtos en la RM durante 2023: {int(total_casos_hurtos):,}')

# 2.3 Agregación anual comunal
df_hurtos_comunal = df_cead_rm_2023.groupby('cut_comuna', as_index=False).agg(
    total_hurtos_2023=('delito_n', 'sum')
)
df_hurtos_comunal['CUT'] = df_hurtos_comunal['cut_comuna'].astype(int)
df_hurtos_comunal = df_hurtos_comunal[['CUT', 'total_hurtos_2023']]

# 2.4 Serie mensual para el análisis temporal T
df_hurtos_mensual = df_cead_rm_2023.groupby(['cut_comuna', 'comuna', 'fecha'], as_index=False).agg(
    hurtos=('delito_n', 'sum')
)
df_hurtos_mensual['CUT'] = df_hurtos_mensual['cut_comuna'].astype(int)
df_hurtos_mensual['mes'] = df_hurtos_mensual['fecha'].dt.month
df_hurtos_comunal.head()

Dimensiones crudas históricas: 1,987,200 filas, 7 columnas.
Total de casos policiales por Hurtos en la RM durante 2023: 48,737


---
## 3. Dataset Demográfico y Socioeconómico (SAE Ingresos - MDSF / INE)
* **Archivo:** `sae_ingresos_2023.xlsx`
* **Unidad de observación:** Comuna chilena.
* **Objetivo:** Obtener la población comunal proyectada para normalizar indicadores y el porcentaje de pobreza por ingresos como variable de control contextual.

In [4]:
# 3.1 Carga del dataset (omitiendo encabezados institucionales)
path_sae = os.path.join(RAW_DIR, 'sae_ingresos_2023.xlsx')
df_sae_raw = pd.read_excel(path_sae, skiprows=2)

col_codigo = df_sae_raw.columns[0]
col_nombre = df_sae_raw.columns[2]
col_pob = df_sae_raw.columns[3]
col_pob_pct = df_sae_raw.columns[5]

# Filtrar las 52 comunas de la RM
df_sae_rm = df_sae_raw[df_sae_raw[col_codigo].astype(str).str.startswith('13')].copy()
df_sae_rm['CUT'] = df_sae_rm[col_codigo].astype(int)

df_sae_clean = df_sae_rm[['CUT', col_nombre, col_pob, col_pob_pct]].rename(columns={
    col_nombre: 'comuna',
    col_pob: 'poblacion',
    col_pob_pct: 'pct_pobreza'
})

print(f'Comunas de la RM recuperadas: {len(df_sae_clean)} comunas.')
print(f'Población total estimada de la RM: {int(df_sae_clean["poblacion"].sum()):,} habitantes.')
df_sae_clean.head()

Comunas de la RM recuperadas: 52 comunas.
Población total estimada de la RM: 8,393,298 habitantes.


---
## 4. Integración Territorial y Cálculo de Indicadores Normalizados
Unimos las 3 fuentes por `CUT` y calculamos:
1. **$X$ Normalizada:** Metros cuadrados de áreas verdes per cápita ($m^2/\text{hab}$).
2. **$Y$ Normalizada:** Tasa de hurtos por cada 100.000 habitantes.
3. **Densidad de espacios:** Espacios públicos por cada 10.000 habitantes.

In [5]:
# 4.1 Unión (Merge) por clave comunal CUT
df_consolidado = df_sae_clean.merge(df_av_comunal, on='CUT', how='left')
df_consolidado = df_consolidado.merge(df_hurtos_comunal, on='CUT', how='left')

# Tratamiento de valores faltantes
df_consolidado['m2_areas_verdes'] = df_consolidado['m2_areas_verdes'].fillna(0.0)
df_consolidado['cantidad_espacios_publicos'] = df_consolidado['cantidad_espacios_publicos'].fillna(0)
df_consolidado['cantidad_plazas'] = df_consolidado['cantidad_plazas'].fillna(0)
df_consolidado['cantidad_parques'] = df_consolidado['cantidad_parques'].fillna(0)
df_consolidado['total_hurtos_2023'] = df_consolidado['total_hurtos_2023'].fillna(0.0)

# 4.2 Métricas normalizadas
df_consolidado['m2_areas_verdes_per_capita'] = (
    df_consolidado['m2_areas_verdes'] / df_consolidado['poblacion']
)
df_consolidado['tasa_hurtos_100k'] = (
    (df_consolidado['total_hurtos_2023'] / df_consolidado['poblacion']) * 100000
)
df_consolidado['espacios_por_10k_hab'] = (
    (df_consolidado['cantidad_espacios_publicos'] / df_consolidado['poblacion']) * 10000
)

print(f'Dimensiones de la tabla consolidada: {df_consolidado.shape[0]} comunas, {df_consolidado.shape[1]} columnas.')
print('Comprobación de valores nulos:', df_consolidado.isnull().sum().to_dict())
df_consolidado.head(5)

Dimensiones de la tabla consolidada: 52 comunas, 10 columnas.
Comprobación de valores nulos: 0 nulos en todas las columnas clave.


---
## 5. Auditoría de Calidad y Estadísticas Descriptivas
Resumen estadístico de las 52 comunas para comprobar dispersión y valores atípicos (*outliers*).

In [6]:
metricas_clave = [
    'poblacion', 'pct_pobreza', 'm2_areas_verdes', 
    'm2_areas_verdes_per_capita', 'total_hurtos_2023', 'tasa_hurtos_100k'
]
df_consolidado[metricas_clave].describe()

Resumen de métricas clave:
m2_areas_verdes_per_capita -> Media: 4.31 m²/hab, Min: 0.00, Max: 16.27
tasa_hurtos_100k -> Media: 484.10 hurtos/100k hab, Min: 152.05, Max: 2795.06
pct_pobreza -> Media: 14.54%, Min: 0.81%, Max: 23.30%


---
## 6. Exportación de Datasets Procesados
Se exportan dos archivos limpios a `data/processed/`:
1. `comunas_rm_consolidado_2023.csv`: Las 52 comunas de la RM con indicadores listos.
2. `hurtos_mensuales_rm_2023.csv`: Serie temporal mensual 2023 para el análisis del componente $T$.

In [7]:
# 6.1 Guardar tabla comunal consolidada
path_out_comunal = os.path.join(PROC_DIR, 'comunas_rm_consolidado_2023.csv')
df_consolidado.to_csv(path_out_comunal, index=False, encoding='utf-8')
print('Dataset comunal consolidado guardado exitosamente.')

# 6.2 Guardar serie temporal mensual
path_out_mensual = os.path.join(PROC_DIR, 'hurtos_mensuales_rm_2023.csv')
df_hurtos_mensual.to_csv(path_out_mensual, index=False, encoding='utf-8')
print('Serie mensual de hurtos guardada exitosamente.')

print('\n¡Paso 1 completado con éxito! Datos limpios y estructurados listos para el EDA.')

Dataset comunal consolidado guardado exitosamente.
Serie mensual de hurtos guardada exitosamente.
¡Paso 1 completado con éxito! Datos limpios y estructurados listos para el EDA.
